In [49]:
import re
import ast
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
import joblib

RAW = Path("../Dataset/Raw")
PROCESSED = Path("../Dataset/Processed")
REPORTS = Path("../Reports")
REPORTS.mkdir(parents=True, exist_ok=True)

print("RAW:", RAW.resolve())
print("PROCESSED:", PROCESSED.resolve())
print("REPORTS:", REPORTS.resolve())

RAW: C:\CSIE\My Projects\Data_Science\BoardGames_Analyzer\Dataset\Raw
PROCESSED: C:\CSIE\My Projects\Data_Science\BoardGames_Analyzer\Dataset\Processed
REPORTS: C:\CSIE\My Projects\Data_Science\BoardGames_Analyzer\Reports


In [50]:
# -----------------------------
# Robust helpers for list fields
# -----------------------------
def to_list_safe(x):
    """
    Convert many possible cell formats into a clean Python list.
    Handles:
    - real lists
    - tuples / sets
    - NaN / None
    - stringified lists like "['Horror', 'Mystery']"
    - pipe/comma separated strings
    """
    if x is None:
        return []

    if isinstance(x, float) and pd.isna(x):
        return []

    if isinstance(x, list):
        return [str(i).strip() for i in x if str(i).strip()]

    if isinstance(x, (tuple, set)):
        return [str(i).strip() for i in x if str(i).strip()]

    if isinstance(x, str):
        s = x.strip()
        if not s or s.lower() == "nan":
            return []

        # Parse stringified Python list/tuple/set
        if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, (list, tuple, set)):
                    return [str(i).strip() for i in parsed if str(i).strip()]
            except Exception:
                pass

        # Split common delimiters
        if "|" in s:
            return [i.strip() for i in s.split("|") if i.strip()]
        if "," in s:
            return [i.strip() for i in s.split(",") if i.strip()]

        return [s]

    return [str(x).strip()]


def normalize_token(s):
    return str(s).strip().lower()


def normalize_list(x):
    return [normalize_token(i) for i in to_list_safe(x) if normalize_token(i)]


def overlap_items(candidate_values, user_values):
    """
    Return overlap list preserving original candidate text as much as possible.
    """
    cand_raw = to_list_safe(candidate_values)
    user_norm = set(normalize_list(user_values))

    out = []
    seen = set()

    for item in cand_raw:
        norm = normalize_token(item)
        if norm in user_norm and norm not in seen:
            out.append(str(item).strip())
            seen.add(norm)

    return out


def overlap_score(candidate_values, user_values):
    """
    Normalized overlap score in [0, 1]
    denominator = number of user requested traits
    """
    user_norm = set(normalize_list(user_values))
    if not user_norm:
        return 0.0

    cand_norm = set(normalize_list(candidate_values))
    hit = len(cand_norm.intersection(user_norm))
    return hit / len(user_norm)

def coalesce_list_columns(df, base_name):
    """
    Merge duplicate columns created by pd.merge into one clean list column.

    Supports suffix patterns:
    - base_name
    - base_name_x / base_name_y
    - base_name_title / base_name_trait
    """
    candidates = [
        c for c in [
            base_name,
            f"{base_name}_x",
            f"{base_name}_y",
            f"{base_name}_title",
            f"{base_name}_trait",
        ]
        if c in df.columns
    ]

    if not candidates:
        df[base_name] = [[] for _ in range(len(df))]
        return df

    def combine_row(row):
        merged = []
        seen = set()

        for c in candidates:
            for item in to_list_safe(row[c]):
                norm = normalize_token(item)
                if norm and norm not in seen:
                    merged.append(str(item).strip())
                    seen.add(norm)

        return merged

    df[base_name] = df.apply(combine_row, axis=1)

    for c in candidates:
        if c != base_name and c in df.columns:
            df.drop(columns=c, inplace=True)

    return df

In [51]:
meta_path = RAW / "games_detailed_info.csv"
df_meta = pd.read_csv(meta_path, low_memory=False)

if "id" not in df_meta.columns:
    raise ValueError("games_detailed_info.csv must contain column 'id'")

df_meta["id"] = pd.to_numeric(df_meta["id"], errors="coerce")
df_meta = df_meta.dropna(subset=["id"]).copy()
df_meta["id"] = df_meta["id"].astype(int)

# indexed metadata
df_meta = df_meta.drop_duplicates(subset=["id"]).set_index("id").copy()

# canonical working dataframe for discovery functions
games = df_meta.reset_index().copy()

print("df_meta shape:", df_meta.shape)
print("games shape:", games.shape)
display(games.head(3))

df_meta shape: (21631, 55)
games shape: (21631, 56)


,id,Unnamed: 0,type,thumbnail,image,primary,alternate,description,yearpublished,minplayers,...,War Game Rank,Customizable Rank,Children's Game Rank,RPG Item Rank,Accessory Rank,Video Game Rank,Amiga Rank,Commodore 64 Rank,Arcade Rank,Atari ST Rank
0,30549,0,boardgame,https://cf.geekdo-images.com/S3ybV1LAp-8SnHIXL...,https://cf.geekdo-images.com/S3ybV1LAp-8SnHIXL...,Pandemic,"['EPIZOotic', 'Pandemia', 'Pandemia 10 Anivers...","In Pandemic, several virulent diseases have br...",2008,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,822,1,boardgame,https://cf.geekdo-images.com/okM0dq_bEXnbyQTOv...,https://cf.geekdo-images.com/okM0dq_bEXnbyQTOv...,Carcassonne,"['Carcassonne Jubilee Edition', 'Carcassonne: ...",Carcassonne is a tile-placement game in which ...,2000,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,13,2,boardgame,https://cf.geekdo-images.com/W3Bsga_uLP9kO91gZ...,https://cf.geekdo-images.com/W3Bsga_uLP9kO91gZ...,Catan,"['CATAN', 'Catan (Колонизаторы)', 'Catan telep...","In CATAN (formerly The Settlers of Catan), pla...",1995,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [52]:
# -----------------------------
# Canonical column names
# -----------------------------

print("All columns in games:")
print(list(games.columns))

# Try to auto-detect the title/name column
possible_name_cols = ["name", "primary", "title", "game_name", "Name"]

detected_name_col = None
for c in possible_name_cols:
    if c in games.columns:
        detected_name_col = c
        break

if detected_name_col is None:
    raise ValueError(
        "Could not find a valid game title column. "
        f"Available columns are: {list(games.columns)}"
    )

NAME_COL = detected_name_col
MECH_COL = "boardgamemechanic"
THEME_COL = "boardgamecategory"
FAMILY_COL = "boardgamefamily"
PUB_COL = "boardgamepublisher"

NUM_COLS = {
    "minplayers": "minplayers",
    "maxplayers": "maxplayers",
    "minplaytime": "minplaytime",
    "maxplaytime": "maxplaytime",
    "minage": "minage",
    "avgweight": "averageweight",
    "usersrated": "usersrated",
}

print("Detected NAME_COL:", NAME_COL)

# warnings for missing metadata columns
for c in [NAME_COL, MECH_COL, THEME_COL, FAMILY_COL, PUB_COL]:
    if c not in df_meta.columns:
        print(f"WARNING: missing column in df_meta: {c}")

# harmonize numeric fields on games dataframe
if "averageweight" in games.columns:
    games["complexity"] = pd.to_numeric(games["averageweight"], errors="coerce")
else:
    games["complexity"] = np.nan

if "usersrated" in games.columns:
    games["num_votes"] = pd.to_numeric(games["usersrated"], errors="coerce")
else:
    games["num_votes"] = np.nan

if "bayesaverage" in games.columns:
    games["avg_rating"] = pd.to_numeric(games["bayesaverage"], errors="coerce")
elif "average" in games.columns:
    games["avg_rating"] = pd.to_numeric(games["average"], errors="coerce")
else:
    games["avg_rating"] = np.nan

if "yearpublished" in games.columns:
    games["yearpublished"] = pd.to_numeric(games["yearpublished"], errors="coerce")
else:
    games["yearpublished"] = np.nan

display(games[[c for c in ["id", NAME_COL, "avg_rating", "complexity", "num_votes", "yearpublished"] if c in games.columns]].head(5))

All columns in games:
['id', 'Unnamed: 0', 'type', 'thumbnail', 'image', 'primary', 'alternate', 'description', 'yearpublished', 'minplayers', 'maxplayers', 'suggested_num_players', 'suggested_playerage', 'suggested_language_dependence', 'playingtime', 'minplaytime', 'maxplaytime', 'minage', 'boardgamecategory', 'boardgamemechanic', 'boardgamefamily', 'boardgameexpansion', 'boardgameimplementation', 'boardgamedesigner', 'boardgameartist', 'boardgamepublisher', 'usersrated', 'average', 'bayesaverage', 'Board Game Rank', 'Strategy Game Rank', 'Family Game Rank', 'stddev', 'median', 'owned', 'trading', 'wanting', 'wishing', 'numcomments', 'numweights', 'averageweight', 'boardgameintegration', 'boardgamecompilation', 'Party Game Rank', 'Abstract Game Rank', 'Thematic Rank', 'War Game Rank', 'Customizable Rank', "Children's Game Rank", 'RPG Item Rank', 'Accessory Rank', 'Video Game Rank', 'Amiga Rank', 'Commodore 64 Rank', 'Arcade Rank', 'Atari ST Rank']
Detected NAME_COL: primary


,id,primary,avg_rating,complexity,num_votes,yearpublished
0,30549,Pandemic,7.48669,2.4063,109006,2008
1,822,Carcassonne,7.30857,1.9057,108776,2000
2,13,Catan,6.96965,2.3130,108064,1995
3,68448,7 Wonders,7.63355,2.3264,90021,2010
4,36218,Dominion,7.49912,2.3542,81582,2008


In [53]:
TRAIT_COLUMNS = [MECH_COL, THEME_COL, FAMILY_COL, PUB_COL]

# Create canonical output title column
games["name"] = games[NAME_COL].astype(str)

for col in TRAIT_COLUMNS:
    if col in games.columns:
        games[col] = games[col].apply(to_list_safe)

# keep df_meta aligned too
df_meta_clean = games.set_index("id").copy()

print("Trait columns cleaned.")
print("Canonical title column created: name <-", NAME_COL)

for col in TRAIT_COLUMNS:
    if col in games.columns:
        print(col, "sample:", games[col].iloc[0][:5] if len(games) > 0 else [])

Trait columns cleaned.
Canonical title column created: name <- primary
boardgamemechanic sample: ['Action Points', 'Cooperative Game', 'Hand Management', 'Point to Point Movement', 'Set Collection']
boardgamecategory sample: ['Medical']
boardgamefamily sample: ['Components: Map (Global Scale)', 'Components: Multi-Use Cards', 'Digital Implementations: Board Game Arena', 'Game: Pandemic', 'Medical: Diseases']
boardgamepublisher sample: ['Z-Man Games', 'Albi', 'Asmodee', 'Asmodee Italia', 'Asterion Press']


In [54]:
def norm_token(s: str) -> str:
    return re.sub(r"\s+", " ", str(s).strip().lower())

def to_tokens(x):
    if x is None:
        return []

    if isinstance(x, float) and pd.isna(x):
        return []

    if isinstance(x, list):
        return [norm_token(t) for t in x if str(t).strip()]

    s = str(x).strip()
    if not s:
        return []

    s = s.replace("[", "").replace("]", "").replace("'", "").replace('"', "")

    if "|" in s:
        parts = s.split("|")
    elif "," in s:
        parts = s.split(",")
    else:
        parts = [s]

    return [norm_token(p) for p in parts if str(p).strip()]


def get_tokens_series(df, col):
    if col not in df.columns:
        return pd.Series([[]] * len(df), index=df.index)
    return df[col].apply(to_tokens)


mech_tokens = get_tokens_series(df_meta_clean, MECH_COL)
cat_tokens = get_tokens_series(df_meta_clean, THEME_COL)
fam_tokens = get_tokens_series(df_meta_clean, FAMILY_COL)
pub_tokens = get_tokens_series(df_meta_clean, PUB_COL)

print("Using NAME_COL =", NAME_COL)

for gid in [13, 30549, 822]:
    if gid in df_meta_clean.index:
        game_title = df_meta_clean.loc[gid, NAME_COL]
        print(gid, game_title)
        print("  mech:", mech_tokens.loc[gid][:8])
        print("  cat :", cat_tokens.loc[gid][:8])

Using NAME_COL = primary
13 Catan
  mech: ['dice rolling', 'hexagon grid', 'income', 'modular board', 'network and route building', 'race', 'random production', 'trading']
  cat : ['economic', 'negotiation']
30549 Pandemic
  mech: ['action points', 'cooperative game', 'hand management', 'point to point movement', 'set collection', 'trading', 'variable player powers']
  cat : ['medical']
822 Carcassonne
  mech: ['area majority / influence', 'map addition', 'tile placement']
  cat : ['city building', 'medieval', 'territory building']


In [55]:
desc_sim = joblib.load(PROCESSED / "desc_topk_csr.joblib")
content_sim = joblib.load(PROCESSED / "content_topk_csr.joblib")
emb_sim = joblib.load(PROCESSED / "emb_topk_csr.joblib")

meta_ids = df_meta.index.to_numpy()
id_to_idx = {int(gid): i for i, gid in enumerate(meta_ids)}
idx_to_id = {i: int(gid) for i, gid in enumerate(meta_ids)}

print("Similarity matrices loaded.")
print("Universe size:", len(meta_ids))

Similarity matrices loaded.
Universe size: 21631


In [56]:
sent_path = PROCESSED / "sentiment_summary.csv"
sent_summary = pd.read_csv(sent_path, index_col="id") if sent_path.exists() else None

sent_mean = None
if sent_summary is not None and "avg_sentiment_score" in sent_summary.columns:
    sent_summary.index = pd.to_numeric(sent_summary.index, errors="coerce")
    sent_summary = sent_summary[~pd.isna(sent_summary.index)].copy()
    sent_summary.index = sent_summary.index.astype(int)
    sent_mean = float(sent_summary["avg_sentiment_score"].mean())
else:
    sent_summary = None

if sent_summary is not None:
    games["sentiment_score"] = games["id"].map(sent_summary["avg_sentiment_score"]).fillna(sent_mean)
else:
    games["sentiment_score"] = 0.0

print("Sentiment loaded:", sent_summary is not None)
print("sent_mean:", sent_mean)

Sentiment loaded: True
sent_mean: 0.9701084770133322


In [57]:
ratings_path = RAW / "large_user_ratings.csv"
ratings_df = pd.read_csv(ratings_path)

USER_COL = "Username"
ITEM_COL = "BGGId"
RATING_COL = "Rating"

ratings_df = ratings_df.dropna(subset=[USER_COL, ITEM_COL, RATING_COL]).copy()
ratings_df[ITEM_COL] = pd.to_numeric(ratings_df[ITEM_COL], errors="coerce")
ratings_df = ratings_df.dropna(subset=[ITEM_COL]).copy()
ratings_df[ITEM_COL] = ratings_df[ITEM_COL].astype(int)

pop_counts = ratings_df.groupby(ITEM_COL).size()
pop_counts = pop_counts.reindex(meta_ids).fillna(0).astype(int)

pop_norm = (pop_counts - pop_counts.min()) / (pop_counts.max() - pop_counts.min() + 1e-9)
pop_norm = pop_norm.fillna(0.0)

games["popularity_norm"] = games["id"].map(pop_norm).fillna(0.0)

print("Popularity ready.")
print("Ratings rows:", len(ratings_df))

Popularity ready.
Ratings rows: 18942152


In [58]:
def safe_minmax(series, full_index):
    s = pd.Series(series).reindex(full_index).astype(float)
    arr = s.values

    if len(arr) == 0:
        return pd.Series(dtype=float)

    if np.all(np.isnan(arr)):
        return pd.Series(0.0, index=full_index)

    mn = np.nanmin(arr)
    mx = np.nanmax(arr)

    if mx == mn:
        return pd.Series(0.0, index=full_index)

    arr = np.nan_to_num(arr, nan=np.nanmean(arr))
    norm = (arr - mn) / (mx - mn + 1e-9)
    return pd.Series(norm, index=full_index)


def topk_from_sim(seed_ids, sim_csr, k=1200, strategy="mean"):
    seed_idxs = [id_to_idx[sid] for sid in seed_ids if sid in id_to_idx]
    if not seed_idxs:
        return pd.Series(dtype=float)

    scores = defaultdict(float)
    counts = defaultdict(int)

    for si in seed_idxs:
        row = sim_csr.getrow(si)
        for c, v in zip(row.indices, row.data):
            gid = idx_to_id[c]
            if strategy == "max":
                scores[gid] = max(scores[gid], float(v))
            else:
                scores[gid] += float(v)
                counts[gid] += 1

    if strategy != "max":
        for gid in list(scores.keys()):
            scores[gid] /= max(counts[gid], 1)

    for sid in seed_ids:
        scores.pop(int(sid), None)

    items = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]
    return pd.Series(dict(items), dtype=float)

In [59]:
df_names = df_meta_clean[[NAME_COL]].copy()
df_names["name_norm"] = df_names[NAME_COL].fillna("").apply(norm_token)

def search_titles(query, topn=10):
    q = norm_token(query)
    if not q:
        return pd.DataFrame(columns=["id", "name"])

    exact = df_names[df_names["name_norm"] == q].reset_index()[["id", NAME_COL]]
    if len(exact) > 0:
        exact.columns = ["id", "name"]
        return exact.head(topn)

    cand = df_names[df_names["name_norm"].str.contains(re.escape(q), na=False)].reset_index()[["id", NAME_COL]]
    cand.columns = ["id", "name"]

    if "usersrated" in df_meta_clean.columns:
        cand = cand.merge(df_meta_clean[["usersrated"]], left_on="id", right_index=True, how="left")
        cand["usersrated"] = pd.to_numeric(cand["usersrated"], errors="coerce").fillna(0)
        cand = cand.sort_values("usersrated", ascending=False).drop(columns=["usersrated"])

    return cand.head(topn)


def resolve_titles_to_ids(titles, pick="auto"):
    seed_ids = []
    candidates_info = []

    for t in titles:
        matches = search_titles(t, topn=10)

        if len(matches) == 0:
            candidates_info.append((t, []))
            continue

        ids = matches["id"].tolist()
        candidates_info.append((t, ids))

        if pick == "all":
            seed_ids.extend(ids)
        else:
            seed_ids.append(int(ids[0]))

    seen = set()
    uniq = []
    for x in seed_ids:
        if int(x) not in seen:
            uniq.append(int(x))
            seen.add(int(x))

    return uniq, candidates_info


search_titles("splendor", topn=5)

,id,name
0,148228,Splendor


In [60]:
DIFF_BUCKETS = {
    "low": (None, 2.0),
    "medium": (2.0, 3.0),
    "high": (3.0, None),
}

TYPE_SYNONYMS = {
    "co-op": {"mechanic": ["cooperative game"]},
    "coop": {"mechanic": ["cooperative game"]},
    "cooperative": {"mechanic": ["cooperative game"]},
    "competitive": {"exclude_mechanic": ["cooperative game"]},
    "mystery": {"category": ["mystery"]},
    "horror": {"category": ["horror"]},
    "farming": {"category": ["farming"]},
    "fun": {"category": ["party game", "humor"]},
    "family": {"family": ["family games", "children's games"]},
    "strategy": {"category": ["strategy"]},
}

def parse_user_types(type_inputs):
    include_mech, exclude_mech, include_cat, include_family = [], [], [], []
    difficulty = None

    for raw in (type_inputs or []):
        t = norm_token(raw)

        if t in ("low", "medium", "high"):
            difficulty = t
            continue

        rule = TYPE_SYNONYMS.get(t)
        if rule:
            include_mech += rule.get("mechanic", [])
            exclude_mech += rule.get("exclude_mechanic", [])
            include_cat += rule.get("category", [])
            include_family += rule.get("family", [])
        else:
            include_cat.append(t)

    def dedupe(lst):
        out, seen = [], set()
        for x in lst:
            x = norm_token(x)
            if x and x not in seen:
                out.append(x)
                seen.add(x)
        return out

    return {
        "include_mech": dedupe(include_mech),
        "exclude_mech": dedupe(exclude_mech),
        "include_cat": dedupe(include_cat),
        "include_family": dedupe(include_family),
        "difficulty": difficulty
    }

In [61]:
def recommend_by_preferences(
    types=None,
    min_players=None, max_players=None,
    min_playtime=None, max_playtime=None,
    max_age=None, min_age=None,
    topk=20,
    w_trait=0.65, w_sent=0.10, w_pop=0.25,
    require_any=True
):
    rules = parse_user_types(types)
    inc_mech = set(rules["include_mech"])
    exc_mech = set(rules["exclude_mech"])
    inc_cat = set(rules["include_cat"])
    inc_fam = set(rules["include_family"])
    diff = rules["difficulty"]

    candidates = df_meta_clean.copy()

    def apply_range(col, lo=None, hi=None):
        nonlocal candidates
        if col not in candidates.columns:
            return
        s = pd.to_numeric(candidates[col], errors="coerce")
        mask = pd.Series(True, index=candidates.index)
        if lo is not None:
            mask &= (s >= lo)
        if hi is not None:
            mask &= (s <= hi)
        candidates = candidates[mask.fillna(False)]

    apply_range(NUM_COLS["minplayers"], min_players, None)
    apply_range(NUM_COLS["maxplayers"], None, max_players)
    apply_range(NUM_COLS["minplaytime"], min_playtime, None)
    apply_range(NUM_COLS["maxplaytime"], None, max_playtime)
    apply_range(NUM_COLS["minage"], min_age, None)
    apply_range(NUM_COLS["minage"], None, max_age)

    if diff in DIFF_BUCKETS and NUM_COLS["avgweight"] in candidates.columns:
        lo, hi = DIFF_BUCKETS[diff]
        apply_range(NUM_COLS["avgweight"], lo, hi)

    cand_ids = candidates.index.astype(int)

    rows = []
    for gid in cand_ids:
        m = set(mech_tokens.loc[gid])
        c = set(cat_tokens.loc[gid])
        f = set(fam_tokens.loc[gid])

        if exc_mech and len(m & exc_mech) > 0:
            continue

        mech_hit = len(m & inc_mech) if inc_mech else 0
        cat_hit = len(c & inc_cat) if inc_cat else 0
        fam_hit = len(f & inc_fam) if inc_fam else 0

        total_prefs = len(inc_mech) + len(inc_cat) + len(inc_fam)
        score_trait_raw = 0.0 if total_prefs == 0 else (mech_hit + cat_hit + fam_hit) / total_prefs

        if require_any and total_prefs > 0 and (mech_hit + cat_hit + fam_hit) == 0:
            continue

        rows.append((gid, score_trait_raw, mech_hit, cat_hit, fam_hit))

    if len(rows) == 0:
        return pd.DataFrame(columns=[
            "id", "name", "score", "score_trait", "score_sent", "score_pop",
            "matched_mechanics", "matched_categories", "matched_families"
        ])

    trait_df = pd.DataFrame(
        rows,
        columns=["id", "score_trait_raw", "mech_hit", "cat_hit", "fam_hit"]
    ).set_index("id")

    trait_norm = safe_minmax(trait_df["score_trait_raw"], trait_df.index.to_numpy())

    if sent_summary is not None and "avg_sentiment_score" in sent_summary.columns:
        sent = sent_summary["avg_sentiment_score"].reindex(trait_df.index).fillna(sent_mean)
        sent_norm = safe_minmax(sent, trait_df.index.to_numpy())
    else:
        sent_norm = pd.Series(0.0, index=trait_df.index)

    pop = pop_norm.reindex(trait_df.index).fillna(0.0)

    final = w_trait * trait_norm + w_sent * sent_norm + w_pop * pop
    top_ids = final.sort_values(ascending=False).head(topk).index.astype(int).tolist()

    out = []
    for gid in top_ids:
        m = set(mech_tokens.loc[gid])
        c = set(cat_tokens.loc[gid])
        f = set(fam_tokens.loc[gid])

        out.append({
            "id": gid,
            "name": df_meta_clean.loc[gid, NAME_COL] if gid in df_meta_clean.index else "",
            "score": float(final.loc[gid]),
            "score_trait": float(trait_norm.loc[gid]),
            "score_sent": float(sent_norm.loc[gid]),
            "score_pop": float(pop.loc[gid]),
            "matched_mechanics": sorted(list(m & inc_mech))[:8],
            "matched_categories": sorted(list(c & inc_cat))[:8],
            "matched_families": sorted(list(f & inc_fam))[:8],
        })

    return pd.DataFrame(out).sort_values("score", ascending=False).reset_index(drop=True)

In [62]:
def build_type_candidates(
    games_df,
    wanted_categories=None,
    wanted_mechanics=None,
    wanted_families=None,
    wanted_publishers=None,
    min_rating=None,
    max_weight=None,
    difficulty_label=None
):
    wanted_categories = wanted_categories or []
    wanted_mechanics = wanted_mechanics or []
    wanted_families = wanted_families or []
    wanted_publishers = wanted_publishers or []

    df = games_df.copy()

    if THEME_COL in df.columns:
        df["matched_categories"] = df[THEME_COL].apply(lambda x: overlap_items(x, wanted_categories))
        df["score_theme"] = df[THEME_COL].apply(lambda x: overlap_score(x, wanted_categories))
    else:
        df["matched_categories"] = [[] for _ in range(len(df))]
        df["score_theme"] = 0.0

    if MECH_COL in df.columns:
        df["matched_mechanics"] = df[MECH_COL].apply(lambda x: overlap_items(x, wanted_mechanics))
        df["score_mech"] = df[MECH_COL].apply(lambda x: overlap_score(x, wanted_mechanics))
    else:
        df["matched_mechanics"] = [[] for _ in range(len(df))]
        df["score_mech"] = 0.0

    if FAMILY_COL in df.columns:
        df["matched_families"] = df[FAMILY_COL].apply(lambda x: overlap_items(x, wanted_families))
        df["score_family"] = df[FAMILY_COL].apply(lambda x: overlap_score(x, wanted_families))
    else:
        df["matched_families"] = [[] for _ in range(len(df))]
        df["score_family"] = 0.0

    if PUB_COL in df.columns:
        df["matched_publishers"] = df[PUB_COL].apply(lambda x: overlap_items(x, wanted_publishers))
        df["score_publisher"] = df[PUB_COL].apply(lambda x: overlap_score(x, wanted_publishers))
    else:
        df["matched_publishers"] = [[] for _ in range(len(df))]
        df["score_publisher"] = 0.0

    df["score_trait"] = (
        0.40 * df["score_theme"] +
        0.35 * df["score_mech"] +
        0.15 * df["score_family"] +
        0.10 * df["score_publisher"]
    )

    if difficulty_label is not None:
        diff = str(difficulty_label).strip().lower()
        if "complexity" in df.columns and diff in DIFF_BUCKETS:
            lo, hi = DIFF_BUCKETS[diff]
            s = pd.to_numeric(df["complexity"], errors="coerce")
            mask = pd.Series(True, index=df.index)
            if lo is not None:
                mask &= (s >= lo)
            if hi is not None:
                mask &= (s <= hi)
            df = df[mask.fillna(False)].copy()

    if min_rating is not None and "avg_rating" in df.columns:
        df = df[pd.to_numeric(df["avg_rating"], errors="coerce") >= min_rating].copy()

    if max_weight is not None and "complexity" in df.columns:
        df = df[pd.to_numeric(df["complexity"], errors="coerce") <= max_weight].copy()

    trait_cols = ["score_theme", "score_mech", "score_family", "score_publisher"]
    df = df[df[trait_cols].sum(axis=1) > 0].copy()

    return df

In [63]:
def recommend_by_titles(
    query_titles,
    top_n=100,
    w_text=0.35,
    w_emb=0.35,
    w_sent=0.10,
    w_pop=0.20,
    candidate_k=1200,
    pick="auto"
):
    seed_ids, candidates_info = resolve_titles_to_ids(query_titles, pick=pick)
    seed_ids = [sid for sid in seed_ids if sid in id_to_idx]

    if len(seed_ids) == 0:
        return pd.DataFrame({
            "note": ["No seed titles found. Try a different spelling."],
            "seed_title_candidates": [str(candidates_info)]
        })

    desc_s = topk_from_sim(seed_ids, desc_sim, k=candidate_k).reindex(meta_ids).fillna(0.0)
    cont_s = topk_from_sim(seed_ids, content_sim, k=candidate_k).reindex(meta_ids).fillna(0.0)
    emb_s = topk_from_sim(seed_ids, emb_sim, k=candidate_k).reindex(meta_ids).fillna(0.0)

    text_norm = safe_minmax(0.5 * desc_s + 0.5 * cont_s, meta_ids)
    emb_norm = safe_minmax(emb_s, meta_ids)

    if sent_summary is not None and "avg_sentiment_score" in sent_summary.columns:
        sent = sent_summary["avg_sentiment_score"].reindex(meta_ids).fillna(sent_mean)
        sent_norm = safe_minmax(sent, meta_ids)
    else:
        sent_norm = pd.Series(0.0, index=meta_ids)

    pop = pop_norm.reindex(meta_ids).fillna(0.0)

    score_like = w_text * text_norm + w_emb * emb_norm + w_sent * sent_norm + w_pop * pop

    for sid in seed_ids:
        if sid in score_like.index:
            score_like.loc[sid] = -1

    rec_ids = score_like.sort_values(ascending=False).head(top_n).index.astype(int).tolist()

    seed_mech = set().union(*[set(mech_tokens.loc[sid]) for sid in seed_ids if sid in mech_tokens.index]) if len(seed_ids) > 0 else set()
    seed_cat = set().union(*[set(cat_tokens.loc[sid]) for sid in seed_ids if sid in cat_tokens.index]) if len(seed_ids) > 0 else set()
    seed_pub = set().union(*[set(pub_tokens.loc[sid]) for sid in seed_ids if sid in pub_tokens.index]) if len(seed_ids) > 0 else set()

    out = []
    for gid in rec_ids:
        m = set(mech_tokens.loc[gid]) if gid in mech_tokens.index else set()
        c = set(cat_tokens.loc[gid]) if gid in cat_tokens.index else set()
        p = set(pub_tokens.loc[gid]) if gid in pub_tokens.index else set()

        out.append({
            "id": gid,
            "name": df_meta_clean.loc[gid, NAME_COL] if gid in df_meta_clean.index else "",
            "score_like": float(score_like.loc[gid]),
            "score_text": float(text_norm.loc[gid]),
            "score_emb": float(emb_norm.loc[gid]),
            "score_sent_like": float(sent_norm.loc[gid]),
            "score_pop_like": float(pop.loc[gid]),
            "matched_mechanics": sorted(list(m & seed_mech))[:8],
            "matched_categories": sorted(list(c & seed_cat))[:8],
            "matched_publishers": sorted(list(p & seed_pub))[:6],
            "seed_ids_used": seed_ids,
            "seed_title_candidates": str(candidates_info)[:300] + ("..." if len(str(candidates_info)) > 300 else "")
        })

    recs = pd.DataFrame(out).sort_values("score_like", ascending=False).reset_index(drop=True)
    recs["score_like"] = recs["score_like"].fillna(0.0)
    return recs

In [64]:
TRAIT_EXPANSIONS = {
    "strategy": {
        "categories": ["strategy", "economic", "puzzle", "territory building", "city building"],
        "mechanics": ["hand management", "tile placement", "area majority / influence", "set collection"]
    },
    "family": {
        "categories": ["family", "children's game", "party game", "card game"],
        "mechanics": ["tile placement", "set collection", "pattern building"]
    },
    "horror": {
        "categories": ["horror"],
        "mechanics": ["cooperative game", "hidden movement"]
    },
    "mystery": {
        "categories": ["mystery", "deduction"],
        "mechanics": ["deduction", "hidden roles", "memory"]
    },
    "fun": {
        "categories": ["party game", "humor"],
        "mechanics": ["push your luck", "real-time", "voting"]
    },
    "co-op": {
        "categories": [],
        "mechanics": ["cooperative game"]
    },
    "coop": {
        "categories": [],
        "mechanics": ["cooperative game"]
    }
}

In [65]:
def expand_traits(wanted_categories=None, wanted_mechanics=None):
    wanted_categories = wanted_categories or []
    wanted_mechanics = wanted_mechanics or []

    cat_out = list(wanted_categories)
    mech_out = list(wanted_mechanics)

    for item in wanted_categories:
        key = str(item).strip().lower()
        if key in TRAIT_EXPANSIONS:
            cat_out.extend(TRAIT_EXPANSIONS[key].get("categories", []))
            mech_out.extend(TRAIT_EXPANSIONS[key].get("mechanics", []))

    # dedupe
    cat_out = list(dict.fromkeys([str(x).strip() for x in cat_out if str(x).strip()]))
    mech_out = list(dict.fromkeys([str(x).strip() for x in mech_out if str(x).strip()]))

    return cat_out, mech_out

In [66]:
def make_reason(row):
    reasons = []

    cats = to_list_safe(row.get("matched_categories", []))
    mechs = to_list_safe(row.get("matched_mechanics", []))
    fams = to_list_safe(row.get("matched_families", []))
    pubs = to_list_safe(row.get("matched_publishers", []))

    if len(cats) > 0:
        reasons.append("Categories: " + ", ".join(cats))
    if len(mechs) > 0:
        reasons.append("Mechanics: " + ", ".join(mechs))
    if len(fams) > 0:
        reasons.append("Families: " + ", ".join(fams))
    if len(pubs) > 0:
        reasons.append("Publishers: " + ", ".join(pubs))

    if len(reasons) == 0:
        return "No explicit trait match recorded"

    return " | ".join(reasons)


def discover(
    query_titles=None,
    wanted_categories=None,
    wanted_mechanics=None,
    wanted_families=None,
    wanted_publishers=None,
    difficulty_label=None,
    top_n=20,
    alpha_title=0.65,
    alpha_trait=0.35,
    exclude_query_titles=True
):
    """
    Discovery Engine:
    - title-based
    - type-based
    - combined
    """

    query_titles = query_titles or []
    wanted_categories = wanted_categories or []
    wanted_mechanics = wanted_mechanics or []
    wanted_families = wanted_families or []
    wanted_publishers = wanted_publishers or []
    
    wanted_categories, wanted_mechanics = expand_traits(
        wanted_categories=wanted_categories,
        wanted_mechanics=wanted_mechanics
    )

    has_title = len(query_titles) > 0
    has_trait = any([
        len(wanted_categories) > 0,
        len(wanted_mechanics) > 0,
        len(wanted_families) > 0,
        len(wanted_publishers) > 0,
        difficulty_label is not None
    ])

    title_df = None
    if has_title:
        title_df = recommend_by_titles(
            query_titles=query_titles,
            top_n=max(top_n * 5, 100)
        ).copy()

        if "score_like" not in title_df.columns:
            raise ValueError("recommend_by_titles() must return 'score_like' column.")

    trait_df = None
    if has_trait:
        trait_df = build_type_candidates(
            games_df=games,
            wanted_categories=wanted_categories,
            wanted_mechanics=wanted_mechanics,
            wanted_families=wanted_families,
            wanted_publishers=wanted_publishers,
            difficulty_label=difficulty_label
        ).copy()

        keep_cols = [
            "id", "name",
            "matched_categories", "matched_mechanics",
            "matched_families", "matched_publishers",
            "score_theme", "score_mech", "score_family",
            "score_publisher", "score_trait"
        ]
        keep_cols = [c for c in keep_cols if c in trait_df.columns]
        trait_df = trait_df[keep_cols].copy()

    if not has_title and not has_trait:
        raise ValueError("Provide at least one title or one trait.")

    meta_cols = ["id", "name", "avg_rating", "complexity", "yearpublished", "sentiment_score", "num_votes"]
    meta_cols = [c for c in meta_cols if c in games.columns]

    # -------------------------
    # Title-only mode
    # -------------------------
    if has_title and not has_trait:
        out = title_df.copy()
        out = out.merge(games[meta_cols].drop_duplicates("id"), on=["id", "name"], how="left")

        out["score_trait"] = 0.0
        out["final_score"] = out["score_like"]
        out["reason"] = out.apply(make_reason, axis=1)

        out = out.sort_values(
            ["final_score", "score_like", "avg_rating", "num_votes"],
            ascending=[False, False, False, False]
        ).head(top_n).reset_index(drop=True)

        return out

    # -------------------------
    # Trait-only mode
    # -------------------------
    if has_trait and not has_title:
        out = trait_df.copy()
        out = out.merge(games[meta_cols].drop_duplicates("id"), on=["id", "name"], how="left")

        out["score_like"] = 0.0

        # Better tie-breaking
        out["rating_norm"] = safe_minmax(
            pd.to_numeric(out["avg_rating"], errors="coerce").fillna(0.0),
            out.index
        )
        out["votes_norm"] = safe_minmax(
            np.log1p(pd.to_numeric(out["num_votes"], errors="coerce").fillna(0.0)),
            out.index
        )
        out["sent_norm"] = safe_minmax(
            pd.to_numeric(out["sentiment_score"], errors="coerce").fillna(0.0),
            out.index
        )

        out["final_score"] = (
            0.70 * out["score_trait"] +
            0.15 * out["rating_norm"] +
            0.10 * out["votes_norm"] +
            0.05 * out["sent_norm"]
        )

        out["reason"] = out.apply(make_reason, axis=1)

        out = out.sort_values(
            ["final_score", "score_trait", "avg_rating", "num_votes"],
            ascending=[False, False, False, False]
        ).head(top_n).reset_index(drop=True)

        return out

    # -------------------------
    # Combined mode
    # -------------------------
    out = title_df.merge(
        trait_df,
        on=["id", "name"],
        how="outer",
        suffixes=("_title", "_trait")
    ).copy()

    if "score_like" not in out.columns:
        out["score_like"] = 0.0
    out["score_like"] = out["score_like"].fillna(0.0)

    for col in ["matched_categories", "matched_mechanics", "matched_families", "matched_publishers"]:
        out = coalesce_list_columns(out, col)

    for col in ["score_theme", "score_mech", "score_family", "score_publisher", "score_trait"]:
        if col not in out.columns:
            out[col] = 0.0
        out[col] = out[col].fillna(0.0)

    out["score_trait"] = (
        0.40 * out["score_theme"] +
        0.35 * out["score_mech"] +
        0.15 * out["score_family"] +
        0.10 * out["score_publisher"]
    )

    out = out.merge(games[meta_cols].drop_duplicates("id"), on=["id", "name"], how="left")

    if exclude_query_titles:
        query_titles_norm = {str(t).strip().lower() for t in query_titles}
        out = out[~out["name"].astype(str).str.strip().str.lower().isin(query_titles_norm)].copy()

    out["rating_norm"] = safe_minmax(
        pd.to_numeric(out["avg_rating"], errors="coerce").fillna(0.0),
        out.index
    )
    out["votes_norm"] = safe_minmax(
        np.log1p(pd.to_numeric(out["num_votes"], errors="coerce").fillna(0.0)),
        out.index
    )
    out["sent_norm"] = safe_minmax(
        pd.to_numeric(out["sentiment_score"], errors="coerce").fillna(0.0),
        out.index
    )

    out["final_score"] = (
        alpha_title * out["score_like"] +
        alpha_trait * out["score_trait"] +
        0.08 * out["rating_norm"] +
        0.04 * out["votes_norm"] +
        0.03 * out["sent_norm"]
    )

    out["why_matched"] = out.apply(
        lambda row: {
            "categories": row["matched_categories"],
            "mechanics": row["matched_mechanics"],
            "families": row["matched_families"],
            "publishers": row["matched_publishers"]
        },
        axis=1
    )

    out["reason"] = out.apply(make_reason, axis=1)

    out = out.sort_values(
        ["final_score", "score_like", "score_trait", "avg_rating", "num_votes"],
        ascending=[False, False, False, False, False]
    ).head(top_n).reset_index(drop=True)

    return out

In [67]:
print("=== Preference-based recommender sample ===")
display(recommend_by_preferences(types=["Mystery", "Horror", "Medium"], topk=10).head(10))

print("\n=== Title-based recommender sample ===")
display(recommend_by_titles(query_titles=["Splendor", "Azul"], top_n=10).head(10))

=== Preference-based recommender sample ===


,id,name,score,score_trait,score_sent,score_pop,matched_mechanics,matched_categories,matched_families
0,10547,Betrayal at House on the Hill,0.091005,0.0,0.0,0.364022,[],[horror],[]
1,205059,Mansions of Madness: Second Edition,0.065077,0.0,0.0,0.260310,[],[horror],[]
2,100423,Elder Sign,0.053422,0.0,0.0,0.213688,[],[horror],[]
3,37046,Ghost Stories,0.046256,0.0,0.0,0.185022,[],[horror],[]
4,113924,Zombicide,0.039999,0.0,0.0,0.159994,[],[horror],[]
5,29368,Last Night on Earth: The Zombie Game,0.031932,0.0,0.0,0.127728,[],[horror],[]
6,176189,Zombicide: Black Plague,0.031389,0.0,0.0,0.125557,[],[horror],[]
7,146652,Legendary Encounters: An Alien Deck Building Game,0.027494,0.0,0.0,0.109976,[],[horror],[]
8,282524,Horrified,0.021485,0.0,0.0,0.085941,[],[horror],[]
9,20963,Fury of Dracula (Second Edition),0.021035,0.0,0.0,0.084141,[],[horror],[]



=== Title-based recommender sample ===


,id,name,score_like,score_text,score_emb,score_sent_like,score_pop_like,matched_mechanics,matched_categories,matched_publishers,seed_ids_used,seed_title_candidates
0,822,Carcassonne,0.630833,0.0,1.000000,0.811302,0.998515,[tile placement],[],"[filosofia éditions, kaissa chess & games, kor...","[148228, 230802]","[('Splendor', [148228]), ('Azul', [230802])]"
1,13,Catan,0.611436,0.0,0.963639,0.760849,0.990386,[race],[economic],"[asmodee, broadway toys ltd, filosofia édition...","[148228, 230802]","[('Splendor', [148228]), ('Azul', [230802])]"
2,30549,Pandemic,0.592768,0.0,0.921036,0.704058,1.000000,[set collection],[],"[asmodee, asterion press, filosofia éditions, ...","[148228, 230802]","[('Splendor', [148228]), ('Azul', [230802])]"
3,9209,Ticket to Ride,0.586966,0.0,0.991762,1.000000,0.699248,"[card drafting, contracts, end game bonuses, s...",[],"[asmodee china, boardgame space, days of wonde...","[148228, 230802]","[('Splendor', [148228]), ('Azul', [230802])]"
4,68448,7 Wonders,0.576293,0.0,0.983736,0.667774,0.826039,[set collection],"[card game, economic]","[asmodee, asterion press, galápagos jogos, gém...","[148228, 230802]","[('Splendor', [148228]), ('Azul', [230802])]"
5,178900,Codenames,0.534498,0.0,0.944755,0.679988,0.679176,[],[card game],"[asmodee, boardgame space, broadway toys ltd, ...","[148228, 230802]","[('Splendor', [148228]), ('Azul', [230802])]"
6,70323,King of Tokyo,0.529660,0.0,0.957214,0.823732,0.561312,[card drafting],[],"[boardgame space, galápagos jogos, gokids 玩樂小子...","[148228, 230802]","[('Splendor', [148228]), ('Azul', [230802])]"
7,3076,Puerto Rico,0.522111,0.0,0.969211,0.623898,0.602487,[end game bonuses],[economic],"[filosofia éditions, kaissa chess & games, lac...","[148228, 230802]","[('Splendor', [148228]), ('Azul', [230802])]"
8,36218,Dominion,0.511559,0.0,0.916035,0.407977,0.750742,[],[card game],"[filosofia éditions, gém klub kft., hobby japa...","[148228, 230802]","[('Splendor', [148228]), ('Azul', [230802])]"
9,266192,Wingspan,0.503945,0.0,0.909733,0.854015,0.500687,"[card drafting, end game bonuses, set collection]",[card game],"[divercentro, ghenos games, kaissa chess & gam...","[148228, 230802]","[('Splendor', [148228]), ('Azul', [230802])]"


In [68]:
print("=== Combined Mode Demo ===")

test_combined = discover(
    query_titles=["Splendor", "Azul"],
    wanted_categories=["Strategy", "Family"],
    top_n=15
)

cols_to_show = [
    "id", "name",
    "score_like", "score_trait", "final_score",
    "avg_rating", "num_votes",
    "matched_categories", "matched_mechanics",
    "reason"
]

cols_to_show = [c for c in cols_to_show if c in test_combined.columns]
display(test_combined[cols_to_show].head(10))

=== Combined Mode Demo ===


,id,name,score_like,score_trait,final_score,avg_rating,num_votes,matched_categories,matched_mechanics,reason
0,822,Carcassonne,0.630833,0.228889,0.623183,7.30857,108776,"[City Building, Territory Building]","[tile placement, Area Majority / Influence]","Categories: City Building, Territory Building ..."
1,68448,7 Wonders,0.576293,0.273333,0.601113,7.63355,90021,"[card game, economic, City Building]","[set collection, Hand Management]","Categories: card game, economic, City Building..."
2,9209,Ticket to Ride,0.586966,0.140000,0.567450,7.30509,76207,[],"[card drafting, contracts, end game bonuses, s...","Mechanics: card drafting, contracts, end game ..."
3,30549,Pandemic,0.592768,0.140000,0.565797,7.48669,109006,[],"[set collection, Hand Management]","Mechanics: set collection, Hand Management | P..."
4,13,Catan,0.611436,0.044444,0.541287,6.96965,108064,[economic],[race],Categories: economic | Mechanics: race | Publi...
5,266192,Wingspan,0.503945,0.184444,0.529174,7.94358,56146,[card game],"[card drafting, end game bonuses, set collecti...",Categories: card game | Mechanics: card drafti...
6,167791,Terraforming Mars,0.473876,0.298889,0.528530,8.27349,74269,"[economic, Territory Building]","[end game bonuses, set collection, tile placem...","Categories: economic, Territory Building | Mec..."
7,178900,Codenames,0.534498,0.088889,0.507647,7.50765,74456,"[card game, Party Game]",[],"Categories: card game, Party Game | Publishers..."
8,3076,Puerto Rico,0.522111,0.088889,0.500383,7.83732,65455,"[economic, City Building]",[end game bonuses],"Categories: economic, City Building | Mechanic..."
9,36218,Dominion,0.511559,0.114444,0.493886,7.49912,81582,[card game],[Hand Management],Categories: card game | Mechanics: Hand Manage...


In [69]:
print("=== Trait-Only Mode Demo ===")

test2 = discover(
    wanted_categories=["Horror", "Mystery"],
    top_n=15
)

cols_to_show = [
    "id", "name",
    "score_trait", "final_score",
    "avg_rating", "num_votes",
    "matched_categories", "score_theme",
    "reason"
]

cols_to_show = [c for c in cols_to_show if c in test2.columns]
display(test2[cols_to_show].head(10))

=== Trait-Only Mode Demo ===


,id,name,score_trait,final_score,avg_rating,num_votes,matched_categories,score_theme,reason
0,181304,Mysterium,0.406667,0.487523,7.14588,35262,"[Deduction, Horror]",0.666667,"Categories: Deduction, Horror | Mechanics: Coo..."
1,127024,Room 25,0.406667,0.438568,6.42697,6131,"[Deduction, Horror]",0.666667,"Categories: Deduction, Horror | Mechanics: Coo..."
2,181279,Fury of Dracula (Third/Fourth Edition),0.336667,0.428003,7.20338,12479,"[Deduction, Horror]",0.666667,"Categories: Deduction, Horror | Mechanics: Hid..."
3,253214,Escape Tales: The Awakening,0.406667,0.425923,6.42387,2204,"[Deduction, Horror]",0.666667,"Categories: Deduction, Horror | Mechanics: Coo..."
4,147949,One Night Ultimate Werewolf,0.336667,0.425753,6.94652,23184,"[Deduction, Horror]",0.666667,"Categories: Deduction, Horror | Mechanics: Hid..."
5,193037,Dead of Winter: The Long Night,0.336667,0.422867,7.19068,8536,"[Deduction, Horror]",0.666667,"Categories: Deduction, Horror | Mechanics: Coo..."
6,167355,Nemesis,0.273333,0.417784,7.98226,17710,[Horror],0.333333,Categories: Horror | Mechanics: Cooperative Ga...
7,20963,Fury of Dracula (Second Edition),0.336667,0.409359,6.81868,9066,"[Deduction, Horror]",0.666667,"Categories: Deduction, Horror | Mechanics: Hid..."
8,163166,One Night Ultimate Werewolf: Daybreak,0.336667,0.403821,6.85144,5206,"[Deduction, Horror]",0.666667,"Categories: Deduction, Horror | Mechanics: Hid..."
9,150376,Dead of Winter: A Crossroads Game,0.266667,0.400911,7.39192,41405,"[Deduction, Horror]",0.666667,"Categories: Deduction, Horror"


In [70]:
print("=== Title-Only Mode Demo ===")

test_title = discover(
    query_titles=["Splendor", "Azul"],
    top_n=15
)

cols_to_show = [
    "id", "name",
    "score_like", "final_score",
    "avg_rating", "num_votes",
    "matched_categories", "matched_mechanics",
    "reason"
]

cols_to_show = [c for c in cols_to_show if c in test_title.columns]
display(test_title[cols_to_show].head(10))

=== Title-Only Mode Demo ===


,id,name,score_like,final_score,avg_rating,num_votes,matched_categories,matched_mechanics,reason
0,822,Carcassonne,0.630833,0.630833,7.30857,108776,[],[tile placement],Mechanics: tile placement | Publishers: filoso...
1,13,Catan,0.611436,0.611436,6.96965,108064,[economic],[race],Categories: economic | Mechanics: race | Publi...
2,30549,Pandemic,0.592768,0.592768,7.48669,109006,[],[set collection],Mechanics: set collection | Publishers: asmode...
3,9209,Ticket to Ride,0.586966,0.586966,7.30509,76207,[],"[card drafting, contracts, end game bonuses, s...","Mechanics: card drafting, contracts, end game ..."
4,68448,7 Wonders,0.576293,0.576293,7.63355,90021,"[card game, economic]",[set collection],"Categories: card game, economic | Mechanics: s..."
5,178900,Codenames,0.534498,0.534498,7.50765,74456,[card game],[],"Categories: card game | Publishers: asmodee, b..."
6,70323,King of Tokyo,0.529660,0.529660,7.07281,61141,[],[card drafting],Mechanics: card drafting | Publishers: boardga...
7,3076,Puerto Rico,0.522111,0.522111,7.83732,65455,[economic],[end game bonuses],Categories: economic | Mechanics: end game bon...
8,36218,Dominion,0.511559,0.511559,7.49912,81582,[card game],[],Categories: card game | Publishers: filosofia ...
9,266192,Wingspan,0.503945,0.503945,7.94358,56146,[card game],"[card drafting, end game bonuses, set collection]",Categories: card game | Mechanics: card drafti...


In [71]:
bad_rows = test2[
    test2["matched_categories"].apply(lambda x: len(to_list_safe(x)) > 0) &
    (test2["score_trait"] <= 0)
]

print("Rows with matched_categories but zero trait score:", len(bad_rows))
display(bad_rows[["id", "name", "matched_categories", "score_theme", "score_trait"]].head(20))

Rows with matched_categories but zero trait score: 0


,id,name,matched_categories,score_theme,score_trait


In [72]:
def compact_view(df, top_n=10):
    cols = [
        "name", "final_score", "score_like", "score_trait",
        "avg_rating", "num_votes", "reason"
    ]
    cols = [c for c in cols if c in df.columns]

    out = df[cols].copy().head(top_n)

    for c in ["final_score", "score_like", "score_trait", "avg_rating"]:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").round(4)

    if "num_votes" in out.columns:
        out["num_votes"] = pd.to_numeric(out["num_votes"], errors="coerce").fillna(0).astype(int)

    return out

print("=== Compact Combined View ===")
display(compact_view(test_combined, top_n=10))

print("=== Compact Trait View ===")
display(compact_view(test2, top_n=10))

print("=== Compact Title View ===")
display(compact_view(test_title, top_n=10))

=== Compact Combined View ===


,name,final_score,score_like,score_trait,avg_rating,num_votes,reason
0,Carcassonne,0.6232,0.6308,0.2289,7.3086,108776,"Categories: City Building, Territory Building ..."
1,7 Wonders,0.6011,0.5763,0.2733,7.6336,90021,"Categories: card game, economic, City Building..."
2,Ticket to Ride,0.5675,0.5870,0.1400,7.3051,76207,"Mechanics: card drafting, contracts, end game ..."
3,Pandemic,0.5658,0.5928,0.1400,7.4867,109006,"Mechanics: set collection, Hand Management | P..."
4,Catan,0.5413,0.6114,0.0444,6.9696,108064,Categories: economic | Mechanics: race | Publi...
5,Wingspan,0.5292,0.5039,0.1844,7.9436,56146,Categories: card game | Mechanics: card drafti...
6,Terraforming Mars,0.5285,0.4739,0.2989,8.2735,74269,"Categories: economic, Territory Building | Mec..."
7,Codenames,0.5076,0.5345,0.0889,7.5076,74456,"Categories: card game, Party Game | Publishers..."
8,Puerto Rico,0.5004,0.5221,0.0889,7.8373,65455,"Categories: economic, City Building | Mechanic..."
9,Dominion,0.4939,0.5116,0.1144,7.4991,81582,Categories: card game | Mechanics: Hand Manage...


=== Compact Trait View ===


,name,final_score,score_like,score_trait,avg_rating,num_votes,reason
0,Mysterium,0.4875,0.0,0.4067,7.1459,35262,"Categories: Deduction, Horror | Mechanics: Coo..."
1,Room 25,0.4386,0.0,0.4067,6.4270,6131,"Categories: Deduction, Horror | Mechanics: Coo..."
2,Fury of Dracula (Third/Fourth Edition),0.4280,0.0,0.3367,7.2034,12479,"Categories: Deduction, Horror | Mechanics: Hid..."
3,Escape Tales: The Awakening,0.4259,0.0,0.4067,6.4239,2204,"Categories: Deduction, Horror | Mechanics: Coo..."
4,One Night Ultimate Werewolf,0.4258,0.0,0.3367,6.9465,23184,"Categories: Deduction, Horror | Mechanics: Hid..."
5,Dead of Winter: The Long Night,0.4229,0.0,0.3367,7.1907,8536,"Categories: Deduction, Horror | Mechanics: Coo..."
6,Nemesis,0.4178,0.0,0.2733,7.9823,17710,Categories: Horror | Mechanics: Cooperative Ga...
7,Fury of Dracula (Second Edition),0.4094,0.0,0.3367,6.8187,9066,"Categories: Deduction, Horror | Mechanics: Hid..."
8,One Night Ultimate Werewolf: Daybreak,0.4038,0.0,0.3367,6.8514,5206,"Categories: Deduction, Horror | Mechanics: Hid..."
9,Dead of Winter: A Crossroads Game,0.4009,0.0,0.2667,7.3919,41405,"Categories: Deduction, Horror"


=== Compact Title View ===


,name,final_score,score_like,score_trait,avg_rating,num_votes,reason
0,Carcassonne,0.6308,0.6308,0.0,7.3086,108776,Mechanics: tile placement | Publishers: filoso...
1,Catan,0.6114,0.6114,0.0,6.9696,108064,Categories: economic | Mechanics: race | Publi...
2,Pandemic,0.5928,0.5928,0.0,7.4867,109006,Mechanics: set collection | Publishers: asmode...
3,Ticket to Ride,0.5870,0.5870,0.0,7.3051,76207,"Mechanics: card drafting, contracts, end game ..."
4,7 Wonders,0.5763,0.5763,0.0,7.6336,90021,"Categories: card game, economic | Mechanics: s..."
5,Codenames,0.5345,0.5345,0.0,7.5076,74456,"Categories: card game | Publishers: asmodee, b..."
6,King of Tokyo,0.5297,0.5297,0.0,7.0728,61141,Mechanics: card drafting | Publishers: boardga...
7,Puerto Rico,0.5221,0.5221,0.0,7.8373,65455,Categories: economic | Mechanics: end game bon...
8,Dominion,0.5116,0.5116,0.0,7.4991,81582,Categories: card game | Publishers: filosofia ...
9,Wingspan,0.5039,0.5039,0.0,7.9436,56146,Categories: card game | Mechanics: card drafti...


## Notebook 08 Conclusion

This notebook implements a hybrid Board Game Discovery Engine with three modes:

1. Title-based discovery using similarity signals
2. Trait-based discovery using category/mechanic/family/publisher overlap
3. Combined discovery using both title similarity and structured trait preferences

Key improvements in this notebook:
- fixed trait-score loss after merge
- standardized metadata columns for stable merging
- added trait expansion for broader human-friendly queries
- improved trait-only ranking with quality-aware tie-breaking
- added explainable recommendation reasons

The final output supports interpretable board game recommendations rather than black-box ranking only.